# Few-Shot Learning for South Sudan Flood Prediction

**Problem**: Only 70 samples with 11 flood events - perfect for few-shot learning!

**Approach**: Prototypical Networks + Meta-learning (MAML)

In [ ]:
!pip install learn2learn torch

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import learn2learn as l2l
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. Load and Prepare Data for Few-Shot Learning

In [ ]:
# Load data
df = pd.read_csv('south_sudan_flood_training_data.csv')

# Prepare features
features = ['sar_before', 'sar_after', 'sar_difference', 'sar_change',
           'elevation', 'slope', 'water_occurrence', 'river_distance',
           'annual_precipitation', 'pre_flood_precipitation']

X = df[features].values
y = df['flood_label'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset: {X_scaled.shape}, Floods: {y.sum()}/{len(y)}")

## 2. Few-Shot Dataset Class

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create dataset
dataset = FloodDataset(X_scaled, y)

# Split flood/non-flood samples
flood_indices = np.where(y == 1)[0]
no_flood_indices = np.where(y == 0)[0]

print(f"Flood samples: {len(flood_indices)}")
print(f"No-flood samples: {len(no_flood_indices)}")

## 3. Prototypical Network

In [ ]:
class PrototypicalNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 32)
        )
    
    def forward(self, x):
        return self.encoder(x)
    
    def compute_prototypes(self, support_embeddings, support_labels, n_classes=2):
        prototypes = []
        for c in range(n_classes):
            mask = (support_labels == c)
            if mask.sum() > 0:
                prototype = support_embeddings[mask].mean(0)
            else:
                prototype = torch.zeros_like(support_embeddings[0])
            prototypes.append(prototype)
        return torch.stack(prototypes)
    
    def classify(self, query_embeddings, prototypes):
        distances = torch.cdist(query_embeddings, prototypes)
        return -distances  # Negative distance as logits

model = PrototypicalNetwork(input_dim=len(features))
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

## 4. Few-Shot Episode Generation

In [ ]:
def create_episode(X, y, n_support=3, n_query=5):
    """Create a few-shot episode with support and query sets"""
    flood_idx = np.where(y == 1)[0]
    no_flood_idx = np.where(y == 0)[0]
    
    # Sample support set (few examples per class)
    support_flood = np.random.choice(flood_idx, min(n_support, len(flood_idx)), replace=False)
    support_no_flood = np.random.choice(no_flood_idx, n_support, replace=False)
    
    # Sample query set
    remaining_flood = np.setdiff1d(flood_idx, support_flood)
    remaining_no_flood = np.setdiff1d(no_flood_idx, support_no_flood)
    
    query_flood = np.random.choice(remaining_flood, min(n_query//2, len(remaining_flood)), replace=False)
    query_no_flood = np.random.choice(remaining_no_flood, n_query//2, replace=False)
    
    # Combine
    support_indices = np.concatenate([support_flood, support_no_flood])
    query_indices = np.concatenate([query_flood, query_no_flood])
    
    support_x = torch.FloatTensor(X[support_indices])
    support_y = torch.LongTensor(y[support_indices])
    query_x = torch.FloatTensor(X[query_indices])
    query_y = torch.LongTensor(y[query_indices])
    
    return support_x, support_y, query_x, query_y

# Test episode creation
sup_x, sup_y, q_x, q_y = create_episode(X_scaled, y)
print(f"Episode: Support {sup_x.shape}, Query {q_x.shape}")
print(f"Support labels: {sup_y.numpy()}")
print(f"Query labels: {q_y.numpy()}")

## 5. Training Loop

In [ ]:
def train_prototypical(model, X, y, n_episodes=1000, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    accuracies = []
    
    for episode in range(n_episodes):
        # Create episode
        support_x, support_y, query_x, query_y = create_episode(X, y, n_support=2, n_query=4)
        
        # Forward pass
        support_embeddings = model(support_x)
        query_embeddings = model(query_x)
        
        # Compute prototypes
        prototypes = model.compute_prototypes(support_embeddings, support_y)
        
        # Classify queries
        logits = model.classify(query_embeddings, prototypes)
        
        # Loss
        loss = F.cross_entropy(logits, query_y)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        with torch.no_grad():
            pred = logits.argmax(1)
            acc = (pred == query_y).float().mean()
            
        losses.append(loss.item())
        accuracies.append(acc.item())
        
        if episode % 100 == 0:
            print(f"Episode {episode}: Loss={loss.item():.3f}, Acc={acc.item():.3f}")
    
    return losses, accuracies

# Train model
print("Training Prototypical Network...")
losses, accuracies = train_prototypical(model, X_scaled, y)
print(f"Final accuracy: {np.mean(accuracies[-100:]):.3f}")

## 6. MAML (Model-Agnostic Meta-Learning)

In [ ]:
class MAMLClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    
    def forward(self, x):
        return self.net(x)

# Create MAML model
maml_model = MAMLClassifier(len(features))
maml = l2l.algorithms.MAML(maml_model, lr=0.01, first_order=False)
meta_optimizer = torch.optim.Adam(maml.parameters(), lr=0.001)

print("MAML model created")

In [ ]:
def train_maml(maml, meta_optimizer, X, y, n_episodes=500):
    losses = []
    
    for episode in range(n_episodes):
        meta_optimizer.zero_grad()
        
        # Create episode
        support_x, support_y, query_x, query_y = create_episode(X, y, n_support=3, n_query=4)
        
        # Clone model for inner loop
        learner = maml.clone()
        
        # Inner loop: adapt to support set
        support_pred = learner(support_x)
        support_loss = F.cross_entropy(support_pred, support_y)
        learner.adapt(support_loss)
        
        # Outer loop: evaluate on query set
        query_pred = learner(query_x)
        query_loss = F.cross_entropy(query_pred, query_y)
        
        # Meta-update
        query_loss.backward()
        meta_optimizer.step()
        
        losses.append(query_loss.item())
        
        if episode % 50 == 0:
            print(f"Episode {episode}: Loss={query_loss.item():.3f}")
    
    return losses

# Train MAML
print("Training MAML...")
maml_losses = train_maml(maml, meta_optimizer, X_scaled, y)
print("MAML training complete")

## 7. Evaluation

In [ ]:
def evaluate_few_shot(model, X, y, n_episodes=100, model_type='prototypical'):
    accuracies = []
    f1_scores = []
    
    for _ in range(n_episodes):
        support_x, support_y, query_x, query_y = create_episode(X, y, n_support=2, n_query=6)
        
        with torch.no_grad():
            if model_type == 'prototypical':
                support_embeddings = model(support_x)
                query_embeddings = model(query_x)
                prototypes = model.compute_prototypes(support_embeddings, support_y)
                logits = model.classify(query_embeddings, prototypes)
                pred = logits.argmax(1)
            
            elif model_type == 'maml':
                learner = model.clone()
                support_pred = learner(support_x)
                support_loss = F.cross_entropy(support_pred, support_y)
                learner.adapt(support_loss)
                query_pred = learner(query_x)
                pred = query_pred.argmax(1)
            
            acc = (pred == query_y).float().mean().item()
            f1 = f1_score(query_y.numpy(), pred.numpy(), average='weighted')
            
            accuracies.append(acc)
            f1_scores.append(f1)
    
    return np.mean(accuracies), np.std(accuracies), np.mean(f1_scores), np.std(f1_scores)

# Evaluate both models
proto_acc, proto_std, proto_f1, proto_f1_std = evaluate_few_shot(model, X_scaled, y, model_type='prototypical')
maml_acc, maml_std, maml_f1, maml_f1_std = evaluate_few_shot(maml, X_scaled, y, model_type='maml')

print("=== FEW-SHOT LEARNING RESULTS ===")
print(f"Prototypical Network: Acc={proto_acc:.3f}±{proto_std:.3f}, F1={proto_f1:.3f}±{proto_f1_std:.3f}")
print(f"MAML: Acc={maml_acc:.3f}±{maml_std:.3f}, F1={maml_f1:.3f}±{maml_f1_std:.3f}")

## 8. Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Prototypical losses
axes[0].plot(losses)
axes[0].set_title('Prototypical Network Loss')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Loss')

# Prototypical accuracy
axes[1].plot(accuracies)
axes[1].set_title('Prototypical Network Accuracy')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Accuracy')

# MAML losses
axes[2].plot(maml_losses)
axes[2].set_title('MAML Loss')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Loss')

plt.tight_layout()
plt.show()

# Performance comparison
methods = ['Prototypical', 'MAML']
accs = [proto_acc, maml_acc]
f1s = [proto_f1, maml_f1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(methods, accs, color=['blue', 'green'])
axes[0].set_title('Few-Shot Accuracy')
axes[0].set_ylabel('Accuracy')

axes[1].bar(methods, f1s, color=['blue', 'green'])
axes[1].set_title('Few-Shot F1 Score')
axes[1].set_ylabel('F1 Score')

plt.tight_layout()
plt.show()

## Summary

**Few-Shot Learning Benefits for South Sudan Flood Prediction:**

1. **Perfect for Small Datasets**: 70 samples → ideal for few-shot learning
2. **Handles Class Imbalance**: Works with just 2-3 flood examples per episode
3. **Fast Adaptation**: Quickly adapts to new flood patterns
4. **Meta-Learning**: Learns how to learn from limited flood data

**Key Advantages:**
- No need for data augmentation (SMOTE)
- Better generalization with limited data
- Can adapt to new regions with few examples
- Robust to extreme class imbalance